<a href="https://colab.research.google.com/github/grabuffo/BrainStim_ANN_fMRI_HCP/blob/main/notebooks/Fit_TMS_fMRI_data2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Train Subject-Specific ANNs on REST + STIM Sessions Combined (Option A: Per-Session Split)

**Novel Approach:** Train a single MLP per subject on **concatenated rest and stimulation sessions**, using a per-session train/test split.

**Rationale:**
- Rest and stim capture different but complementary states of intrinsic brain dynamics
- Combined training improves model robustness across states
- Per-session splitting ensures clean train/test separation while allowing diverse training data

**Train/Test Strategy (Option A):**
1. For each session (rest or stim): Split 50/50 within the session
2. Concatenate all train samples from all sessions → global training set
3. Concatenate all test samples from all sessions → global test set
4. Train one ANN per subject on combined rest+stim data
5. Evaluate on held-out test samples (no session bleed)

## Setup

In [ ]:
# --- Setup cell ---

# 1️⃣ Mount Google Drive (for data)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 2️⃣ Clone GitHub repository (for code)
import os, sys, subprocess

repo_dir = "/content/BrainStim_ANN_fMRI_HCP"
if not os.path.exists(repo_dir):
    !git clone https://github.com/grabuffo/BrainStim_ANN_fMRI_HCP.git
else:
    print("Repo already exists ✅")

# 3️⃣ Define paths
data_dir = "/content/drive/MyDrive/Colab Notebooks/Brain_Stim_ANN/data"
tms_pkl_path = os.path.join(data_dir, "TMS_fMRI", "dataset_tian50_schaefer400_allruns.pkl")
results_dir = os.path.join(data_dir, "preprocessed_subjects_tms_fmri")

os.makedirs(results_dir, exist_ok=True)

sys.path.append(repo_dir)

# 4️⃣ Import packages
import numpy as np
import pickle
import json
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
from src import NPI
from src.preprocessing_tms_fmri import preprocess_run, make_inputs_targets
import gc
import warnings
warnings.filterwarnings('ignore')

# 5️⃣ Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Running on:", torch.cuda.get_device_name(0))
else:
    print("⚠️  GPU not detected — training will run on CPU.")

print(f"\nDataset path: {tms_pkl_path}")

## Configuration

In [ ]:
# --- Configuration ---

# Model and training hyperparameters
method = "MLP"
ROI_num = 450
using_steps = 3
batch_size = 64
train_set_proportion = 0.5  # 50/50 train/test split
num_epochs = 100
learning_rate = 5e-4
l2_reg = 5e-5

# Preprocessing parameters
remove_initial_trs = 12
low_hz, high_hz = 0.008, 0.08
tr_stim = 2.4
tr_rest = 2.0  # Rest typically has different TR

# Filtering criterion
min_timepoints = 1000  # Exclude subjects with < 1000 timepoints

# FC analysis: use cortical ROIs only (last n Schaefer regions)
n_cortical = 400  # Use 450 for all ROIs, 400 for cortical Schaefer only

print("Training configuration:")
print(f"  Method: {method}")
print(f"  Regions: {ROI_num}")
print(f"  Steps: {using_steps}")
print(f"  Epochs: {num_epochs}")
print(f"  Batch size: {batch_size}")
print(f"  Train/test split: {train_set_proportion:.1%} / {1-train_set_proportion:.1%}")
print(f"  Min timepoints per subject: {min_timepoints}")
print(f"\nPreprocessing:")
print(f"  Remove initial TRs: {remove_initial_trs}")
print(f"  Bandpass filter: [{low_hz}, {high_hz}] Hz")
print(f"  TR rest: {tr_rest} s, TR stim: {tr_stim} s")

## Load Dataset

In [ ]:
# --- Load dataset ---
print("Loading TMS-fMRI dataset...")

with open(tms_pkl_path, "rb") as f:
    dataset = pickle.load(f)

print(f"✅ Loaded {len(dataset)} subjects")

# Identify subjects with both rest and stim data
subject_ids = []
for sub_id in sorted(dataset.keys()):
    has_rest = 'task-rest' in dataset[sub_id] and len(dataset[sub_id]['task-rest']) > 0
    has_stim = 'task-stim' in dataset[sub_id] and len(dataset[sub_id]['task-stim']) > 0
    if has_rest and has_stim:
        subject_ids.append(sub_id)

print(f"✅ Found {len(subject_ids)} subjects with both rest AND stim sessions")
print(f"   Subject IDs: {subject_ids[:5]}... (showing first 5)")

## Prepare Training Data (Option A: Per-Session Split)

For each subject:
1. Load all rest sessions → split each 50/50 → concatenate all train/test
2. Load all stim sessions → split each 50/50 → concatenate all train/test
3. Mix rest and stim train/test sets
4. Train one model on combined data

In [ ]:
# --- Prepare training data per subject (Option A: per-session split) ---
print("="*70)
print("PREPARING TRAINING DATA (REST + STIM, PER-SESSION SPLIT)")
print("="*70)

per_subject_data = {}

for sub_id in subject_ids:
    print(f"\n{sub_id}:")
    
    train_inputs = []
    train_targets = []
    test_inputs = []
    test_targets = []
    
    total_timepoints = 0
    session_breakdown = []
    
    # ---- PROCESS REST SESSIONS ----
    if 'task-rest' in dataset[sub_id]:
        rest_sessions = dataset[sub_id]['task-rest']
        print(f"  Rest sessions: {len(rest_sessions)}")
        
        for sess_idx in sorted(rest_sessions.keys()):
            sess_data = rest_sessions[sess_idx]
            ts = sess_data.get('time series')
            
            if ts is None or len(ts) == 0:
                continue
            
            ts = np.asarray(ts, dtype=np.float32)
            if ts.shape[0] <= remove_initial_trs:
                continue
            
            # Preprocess (per-session)
            ts_drop = ts[remove_initial_trs:, :]
            ts_proc = preprocess_run(
                ts_drop, tr=tr_rest, n_drop=0,
                low=low_hz, high=high_hz, order=2, zscore=True
            )
            
            if ts_proc.shape[0] <= using_steps:
                continue
            
            # Build inputs/targets
            X, Y = make_inputs_targets(ts_proc, steps=using_steps)
            
            if X.shape[0] > 0:
                # Split 50/50 within this session
                split_idx = X.shape[0] // 2
                X_train_sess = X[:split_idx].astype(np.float32)
                Y_train_sess = Y[:split_idx].astype(np.float32)
                X_test_sess = X[split_idx:].astype(np.float32)
                Y_test_sess = Y[split_idx:].astype(np.float32)
                
                train_inputs.append(X_train_sess)
                train_targets.append(Y_train_sess)
                test_inputs.append(X_test_sess)
                test_targets.append(Y_test_sess)
                
                total_timepoints += X.shape[0]
                session_breakdown.append((f'rest_{sess_idx}', X.shape[0]))
    
    # ---- PROCESS STIM SESSIONS ----
    if 'task-stim' in dataset[sub_id]:
        stim_sessions = dataset[sub_id]['task-stim']
        print(f"  Stim sessions: {len(stim_sessions)}")
        
        for sess_idx in sorted(stim_sessions.keys()):
            sess_data = stim_sessions[sess_idx]
            ts = sess_data.get('time series')
            
            if ts is None or len(ts) == 0:
                continue
            
            ts = np.asarray(ts, dtype=np.float32)
            if ts.shape[0] <= remove_initial_trs:
                continue
            
            # Preprocess (per-session)
            ts_drop = ts[remove_initial_trs:, :]
            ts_proc = preprocess_run(
                ts_drop, tr=tr_stim, n_drop=0,
                low=low_hz, high=high_hz, order=2, zscore=True
            )
            
            if ts_proc.shape[0] <= using_steps:
                continue
            
            # Build inputs/targets
            X, Y = make_inputs_targets(ts_proc, steps=using_steps)
            
            if X.shape[0] > 0:
                # Split 50/50 within this session
                split_idx = X.shape[0] // 2
                X_train_sess = X[:split_idx].astype(np.float32)
                Y_train_sess = Y[:split_idx].astype(np.float32)
                X_test_sess = X[split_idx:].astype(np.float32)
                Y_test_sess = Y[split_idx:].astype(np.float32)
                
                train_inputs.append(X_train_sess)
                train_targets.append(Y_train_sess)
                test_inputs.append(X_test_sess)
                test_targets.append(Y_test_sess)
                
                total_timepoints += X.shape[0]
                session_breakdown.append((f'stim_{sess_idx}', X.shape[0]))
    
    # ---- CONCATENATE ALL SESSIONS ----
    if train_inputs and test_inputs:
        if total_timepoints < min_timepoints:
            print(f"⚠️  Insufficient timepoints ({total_timepoints} < {min_timepoints}), SKIP")
            continue
        
        per_subject_data[sub_id] = {
            'train_inputs': np.concatenate(train_inputs, axis=0),
            'train_targets': np.concatenate(train_targets, axis=0),
            'test_inputs': np.concatenate(test_inputs, axis=0),
            'test_targets': np.concatenate(test_targets, axis=0),
            'n_train': sum([X.shape[0] for X in train_inputs]),
            'n_test': sum([X.shape[0] for X in test_inputs]),
            'session_breakdown': session_breakdown,
        }
        
        print(f"✓ {len(session_breakdown)} sessions, {total_timepoints} total samples")
        print(f"  Train: {per_subject_data[sub_id]['n_train']}, Test: {per_subject_data[sub_id]['n_test']}")
    else:
        print(f"✗ No valid sessions for {sub_id}")

print("\n" + "="*70)
print(f"Data prepared for {len(per_subject_data)} subjects (filtered: >= {min_timepoints} timepoints)")
print("="*70)

# Show which subjects passed/failed
passed_subs = list(per_subject_data.keys())
all_subs = subject_ids
failed_subs = [s for s in all_subs if s not in passed_subs]

print(f"\n✅ PASSED ({len(passed_subs)}):")
for sub in passed_subs:
    print(f"   {sub}: {per_subject_data[sub]['n_train']} train, {per_subject_data[sub]['n_test']} test")

if failed_subs:
    print(f"\n❌ FAILED ({len(failed_subs)}) — likely due to < {min_timepoints} timepoints:")
    for sub in failed_subs[:10]:  # Show first 10
        print(f"   {sub}")
    if len(failed_subs) > 10:
        print(f"   ... and {len(failed_subs) - 10} more")

## Train Models

In [ ]:
# --- Train a model for each subject ---
print("="*70)
print("TRAINING MLP MODELS (REST + STIM COMBINED)")
print("="*70)

results = {}
weights_dir = os.path.join(results_dir, f"trained_models_{method}")
os.makedirs(weights_dir, exist_ok=True)

for sub_id in sorted(per_subject_data.keys()):
    print(f"\n🚀 Training model for subject {sub_id} ({method})")
    
    # Get training data (already split per-session, now concatenated)
    X_train = per_subject_data[sub_id]['train_inputs']
    Y_train = per_subject_data[sub_id]['train_targets']
    X_test = per_subject_data[sub_id]['test_inputs']
    Y_test = per_subject_data[sub_id]['test_targets']
    session_breakdown = per_subject_data[sub_id]['session_breakdown']
    
    print(f"   Train: {X_train.shape[0]} samples, Test: {X_test.shape[0]} samples")
    print(f"   Sessions: {session_breakdown}")
    
    # Initialize model
    model = NPI.build_model(method, ROI_num, using_steps)
    
    # Note: train_NN expects to do its own split, but we're providing pre-split data
    # So we'll use it with train_set_proportion=0.5, feeding X_train as combined data
    # Actually, let's use a custom training loop to use our pre-split X_test
    
    # Custom training loop with pre-split test set
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=l2_reg)
    criterion = torch.nn.MSELoss()
    model.to(device)
    
    train_loss_hist = []
    test_loss_hist = []
    
    # Convert to tensors
    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
    Y_train_t = torch.tensor(Y_train, dtype=torch.float32, device=device)
    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
    Y_test_t = torch.tensor(Y_test, dtype=torch.float32, device=device)
    
    # Training loop
    for epoch in range(num_epochs):
        # Shuffle train data
        idx_shuffle = torch.randperm(X_train_t.shape[0], device=device)
        X_train_shuffled = X_train_t[idx_shuffle]
        Y_train_shuffled = Y_train_t[idx_shuffle]
        
        # Train batches
        model.train()
        epoch_loss = 0
        n_batches = 0
        
        for i in range(0, X_train_shuffled.shape[0], batch_size):
            X_batch = X_train_shuffled[i:i+batch_size]
            Y_batch = Y_train_shuffled[i:i+batch_size]
            
            optimizer.zero_grad()
            Y_pred = model(X_batch)
            loss = criterion(Y_pred, Y_batch)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        train_loss = epoch_loss / n_batches
        
        # Evaluate on test set
        model.eval()
        with torch.no_grad():
            Y_pred_test = model(X_test_t)
            test_loss = criterion(Y_pred_test, Y_test_t).item()
        
        train_loss_hist.append(train_loss)
        test_loss_hist.append(test_loss)
    
    # Save model (same format as before: one per subject)
    model_path = os.path.join(weights_dir, f"{sub_id}_{method}.pt")
    torch.save(model, model_path)
    
    # Store results
    results[sub_id] = {
        "train_loss": train_loss_hist,
        "test_loss": test_loss_hist,
        "final_train_loss": float(train_loss_hist[-1]),
        "final_test_loss": float(test_loss_hist[-1]),
        "method": method,
        "ROI_num": ROI_num,
        "using_steps": using_steps,
        "n_train": int(X_train.shape[0]),
        "n_test": int(X_test.shape[0]),
        "n_sessions": len(session_breakdown),
        "session_breakdown": session_breakdown,
    }
    
    print(f"✅ Done {sub_id}")
    print(f"   Final train loss: {train_loss_hist[-1]:.6f}")
    print(f"   Final test loss: {test_loss_hist[-1]:.6f}")
    print(f"💾 Saved to {model_path}")
    
    # Clean up
    del model
    gc.collect()
    torch.cuda.empty_cache()

# Save summary
results_path = os.path.join(results_dir, f"ANN_results_{method}_restStim.npy")
np.save(results_path, results, allow_pickle=True)

print("\n" + "="*70)
print(f"✅ All {len(results)} subjects trained and saved successfully")
print(f"💾 Results saved to: {weights_dir}")
print("="*70)

## Evaluate Models: Train vs Test Metrics

In [ ]:
# --- Load previously saved results (if skipping training) ---
print("="*70)
print("LOADING RESULTS")
print("="*70)

results_path = os.path.join(results_dir, f"ANN_results_{method}_restStim.npy")
weights_dir = os.path.join(results_dir, f"trained_models_{method}")

# Always recompute MSE from saved models if per_subject_data is available
# This ensures we get accurate MSE values even if results was previously loaded
if len(per_subject_data) > 0 and os.path.exists(weights_dir):
    print(f"📊 Recomputing MSE from {len(per_subject_data)} subjects and saved models...")
    
    criterion = torch.nn.MSELoss()
    reconstructed = {}
    
    for sub_id in sorted(per_subject_data.keys()):
        model_path = os.path.join(weights_dir, f"{sub_id}_{method}.pt")
        
        if os.path.exists(model_path):
            try:
                # Load model
                model = torch.load(model_path, map_location=device, weights_only=False)
                model.eval()
                
                # Get training data
                X_train = per_subject_data[sub_id]['train_inputs']
                Y_train = per_subject_data[sub_id]['train_targets']
                X_test = per_subject_data[sub_id]['test_inputs']
                Y_test = per_subject_data[sub_id]['test_targets']
                
                # Compute MSE losses
                with torch.no_grad():
                    X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
                    Y_train_t = torch.tensor(Y_train, dtype=torch.float32, device=device)
                    Y_train_pred = model(X_train_t)
                    train_loss = float(criterion(Y_train_pred, Y_train_t).item())
                    
                    X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
                    Y_test_t = torch.tensor(Y_test, dtype=torch.float32, device=device)
                    Y_test_pred = model(X_test_t)
                    test_loss = float(criterion(Y_test_pred, Y_test_t).item())
                
                # Preserve training loss history if available
                train_loss_hist = results[sub_id].get("train_loss", []) if sub_id in results else []
                test_loss_hist = results[sub_id].get("test_loss", []) if sub_id in results else []
                
                reconstructed[sub_id] = {
                    "train_loss": train_loss_hist,
                    "test_loss": test_loss_hist,
                    "final_train_loss": train_loss,
                    "final_test_loss": test_loss,
                    "method": method,
                    "ROI_num": ROI_num,
                    "using_steps": using_steps,
                    "n_train": int(per_subject_data[sub_id]['n_train']),
                    "n_test": int(per_subject_data[sub_id]['n_test']),
                    "n_sessions": len(per_subject_data[sub_id]['session_breakdown']),
                    "session_breakdown": per_subject_data[sub_id]['session_breakdown'],
                }
                del model
            except Exception as e:
                print(f"   ⚠️  Error loading {sub_id}: {e}")
    
    results = reconstructed
    print(f"✅ Updated MSE for {len(results)} subjects from saved models\n")

elif len(results) == 0:
    # Only reconstruct if results is empty
    if os.path.exists(results_path):
        print(f"⏭️  Training skipped. Loading previously saved results from:")
        print(f"   {results_path}")
        try:
            results = np.load(results_path, allow_pickle=True).item()
            print(f"✅ Loaded results for {len(results)} subjects")
        except Exception as e:
            print(f"❌ Error loading from {results_path}: {e}")
    else:
        print(f"⚠️  No results in memory and no saved results found.")
        print(f"   Please run the training cell (Cell 8) first.")
else:
    print(f"✅ Using {len(results)} subjects from memory")



In [ ]:
# --- DIAGNOSTIC CHECK ---
print("="*70)
print("DIAGNOSTIC CHECK")
print("="*70)
print(f"\n✓ Data prepared for {len(per_subject_data)} subjects")
print(f"✓ Models trained for {len(results)} subjects")

if len(per_subject_data) == 0:
    print("\n❌ ERROR: per_subject_data is empty!")
    print("   This means no subjects passed the filtering criteria (< 1000 timepoints).")
    print("   Check the 'PREPARING TRAINING DATA' output above.")
    
if len(results) == 0:
    print("\n❌ ERROR: results dictionary is empty!")
    print("   No models were trained successfully.")
    print("   Check the 'TRAINING MLP MODELS' output above.")
    print("   Possible causes: memory issues, dimension mismatches, or training errors.")
else:
    print(f"✓ Training completed successfully for {len(results)} subjects")


In [ ]:
# --- Compute per-region timeseries correlation (train and test) ---
print("="*70)
print("COMPUTING TIMESERIES CORRELATIONS")
print("="*70)

torch.serialization.add_safe_globals([NPI.ANN_MLP, NPI.ANN_CNN, NPI.ANN_RNN, NPI.ANN_VAR])

advanced_metrics = {}

# Guard: skip if no results
if len(per_subject_data) == 0 or len(results) == 0:
    print("\n⏭️  Skipping advanced metrics (no training data or results)")
else:
    for sub_id in sorted(per_subject_data.keys()):
        print(f"\n{sub_id}:")
        
        # Load model
        model_path = os.path.join(weights_dir, f"{sub_id}_{method}.pt")
        model = torch.load(model_path, map_location=device, weights_only=False)
        model.eval()
        
        # Get data
        X_train = per_subject_data[sub_id]['train_inputs']
        Y_train = per_subject_data[sub_id]['train_targets']
        X_test = per_subject_data[sub_id]['test_inputs']
        Y_test = per_subject_data[sub_id]['test_targets']
        
        # Predictions
        with torch.no_grad():
            X_train_t = torch.tensor(X_train, dtype=torch.float32, device=device)
            Y_train_pred = model(X_train_t).cpu().numpy()
            
            X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
            Y_test_pred = model(X_test_t).cpu().numpy()
        
        # Compute per-region timeseries correlation
        train_corr_per_roi = []
        test_corr_per_roi = []
        
        for roi in range(ROI_num):
            # Train
            if np.std(Y_train[:, roi]) > 1e-6 and np.std(Y_train_pred[:, roi]) > 1e-6:
                r = np.corrcoef(Y_train[:, roi], Y_train_pred[:, roi])[0, 1]
                if not np.isnan(r):
                    train_corr_per_roi.append(r)
            
            # Test
            if np.std(Y_test[:, roi]) > 1e-6 and np.std(Y_test_pred[:, roi]) > 1e-6:
                r = np.corrcoef(Y_test[:, roi], Y_test_pred[:, roi])[0, 1]
                if not np.isnan(r):
                    test_corr_per_roi.append(r)
        
        train_mean_corr = np.mean(train_corr_per_roi) if train_corr_per_roi else np.nan
        test_mean_corr = np.mean(test_corr_per_roi) if test_corr_per_roi else np.nan
        
        print(f"  Train timeseries correlation: {train_mean_corr:.4f} (n={len(train_corr_per_roi)} ROIs)")
        print(f"  Test timeseries correlation: {test_mean_corr:.4f} (n={len(test_corr_per_roi)} ROIs)")
        
        # Compute FC correlation (train and test)
        FC_train_emp = np.corrcoef(Y_train.T)
        FC_train_pred = np.corrcoef(Y_train_pred.T)
        
        FC_test_emp = np.corrcoef(Y_test.T)
        FC_test_pred = np.corrcoef(Y_test_pred.T)
        
        # Extract cortical ROIs only (last n_cortical Schaefer regions)
        FC_train_emp_cortical = FC_train_emp[-n_cortical:, -n_cortical:]
        FC_train_pred_cortical = FC_train_pred[-n_cortical:, -n_cortical:]
        FC_test_emp_cortical = FC_test_emp[-n_cortical:, -n_cortical:]
        FC_test_pred_cortical = FC_test_pred[-n_cortical:, -n_cortical:]
        
        # Extract upper triangles
        tri_idx = np.triu_indices(n_cortical, k=1)
        fc_train_corr = np.corrcoef(
            FC_train_emp_cortical[tri_idx],
            FC_train_pred_cortical[tri_idx]
        )[0, 1]
        
        fc_test_corr = np.corrcoef(
            FC_test_emp_cortical[tri_idx],
            FC_test_pred_cortical[tri_idx]
        )[0, 1]
        
        print(f"  Train FC correlation: {fc_train_corr:.4f}")
        print(f"  Test FC correlation: {fc_test_corr:.4f}")
        
        # Store
        advanced_metrics[sub_id] = {
            'train_ts_corr': float(train_mean_corr),
            'test_ts_corr': float(test_mean_corr),
            'train_fc_corr': float(fc_train_corr) if not np.isnan(fc_train_corr) else 0.0,
            'test_fc_corr': float(fc_test_corr) if not np.isnan(fc_test_corr) else 0.0,
        }
        
        del model
        gc.collect()
        torch.cuda.empty_cache()

print("\n" + "="*70)
print(f"✅ Computed advanced metrics for {len(advanced_metrics)} subjects")
print("="*70)

## Compute dFC Variance Correlation (Empirical vs Simulated)

In [ ]:
# --- dFC analysis DISABLED (memory-intensive, not critical) ---
# To enable dFC computation, uncomment the code below.
# However, it may cause RAM crashes on Colab due to large correlation matrices.

"""
def go_edge(tseries):
    # Compute edge time series as product of z-scored signals.
    nregions = tseries.shape[1]
    iTriup = np.triu_indices(nregions, k=1)
    gz = stats.zscore(tseries, axis=0)
    Eseries = gz[:, iTriup[0]] * gz[:, iTriup[1]]
    return Eseries

def compute_dfc_fluidity(tseries):
    # Compute dynamic FC fluidity (variance of edge time series).
    Eseries = go_edge(tseries)
    edge_corr = np.corrcoef(Eseries.T)
    tri_idx = np.triu_indices(edge_corr.shape[0], k=1)
    edge_corr_upper = edge_corr[tri_idx]
    fluidity = np.var(edge_corr_upper)
    del edge_corr, Eseries, edge_corr_upper
    gc.collect()
    return fluidity

dfc_metrics = {}
torch.serialization.add_safe_globals([NPI.ANN_MLP, NPI.ANN_CNN, NPI.ANN_RNN, NPI.ANN_VAR])

for sub_id in sorted(per_subject_data.keys()):
    print(f"\n{sub_id}:")
    model_path = os.path.join(weights_dir, f"{sub_id}_{method}.pt")
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    Y_emp = per_subject_data[sub_id]['test_targets']
    S = using_steps
    N = ROI_num
    init_state = np.zeros((S, N), dtype=np.float32)
    Y_sim = NPI.model_time_series(model, init_state, tlen=Y_emp.shape[0], noise_strength=0.35)
    Y_emp_cortical = Y_emp[:, -n_cortical:]
    Y_sim_cortical = Y_sim[:, -n_cortical:]
    fluidity_emp = compute_dfc_fluidity(Y_emp_cortical)
    fluidity_sim = compute_dfc_fluidity(Y_sim_cortical)
    fluidity_error = abs(fluidity_sim - fluidity_emp) / (fluidity_emp + 1e-8)
    print(f"  Fluidity (empirical): {fluidity_emp:.6f}")
    print(f"  Fluidity (simulated): {fluidity_sim:.6f}")
    print(f"  Fluidity error: {fluidity_error*100:.2f}%")
    dfc_metrics[sub_id] = {
        'fluidity_emp': float(fluidity_emp),
        'fluidity_sim': float(fluidity_sim),
        'fluidity_error_percent': float(fluidity_error * 100),
    }
    del model, Y_sim, Y_emp, Y_emp_cortical, Y_sim_cortical
    gc.collect()
    torch.cuda.empty_cache()

print("\n" + "="*70)
print(f"✅ Computed dFC fluidity metrics for {len(dfc_metrics)} subjects (cortical ROIs only)")
print("="*70)
"""

# dFC analysis disabled to save memory
dfc_metrics = {}
print("⏭️  dFC analysis skipped (commented out to save RAM)")


## Comprehensive Metrics Summary

In [ ]:
# --- Build comprehensive metrics table ---

print("="*70)
print("COMPREHENSIVE METRICS SUMMARY")
print("="*70)

# Debug: Check which dictionaries have data
print(f"\n[DEBUG] Subjects in results: {len(results)}")
print(f"[DEBUG] Subjects in advanced_metrics: {len(advanced_metrics)}")
print(f"[DEBUG] Results keys: {list(results.keys())}")
print(f"[DEBUG] Advanced metrics keys: {list(advanced_metrics.keys())}")

comprehensive_metrics = []

# Only include subjects that are in BOTH results and advanced_metrics
valid_subjects = set(results.keys()) & set(advanced_metrics.keys())
print(f"[DEBUG] Valid subjects (in both dicts): {len(valid_subjects)}")

for sub_id in sorted(valid_subjects):
    row = {
        'Subject': sub_id,
        'N_train': int(results[sub_id]['n_train']),
        'N_test': int(results[sub_id]['n_test']),
        'N_sessions': int(results[sub_id]['n_sessions']),
        'Train_MSE': float(results[sub_id]['final_train_loss']),
        'Test_MSE': float(results[sub_id]['final_test_loss']),
        'Train_TS_Corr': float(advanced_metrics[sub_id]['train_ts_corr']),
        'Test_TS_Corr': float(advanced_metrics[sub_id]['test_ts_corr']),
        'Train_FC_Corr': float(advanced_metrics[sub_id]['train_fc_corr']),
        'Test_FC_Corr': float(advanced_metrics[sub_id]['test_fc_corr']),
    }
    comprehensive_metrics.append(row)

df_metrics = pd.DataFrame(comprehensive_metrics)

# Display
print("\n" + df_metrics.to_string(index=False))

# Save to CSV
csv_path = os.path.join(results_dir, "metrics_rest_stim_combined.csv")
df_metrics.to_csv(csv_path, index=False)
print(f"\n✅ Saved metrics to: {csv_path}")

# Summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)

if len(df_metrics) == 0:
    print("\n⚠️  WARNING: No valid subjects found! DataFrame is empty.")
    print("This may indicate:")
    print("  - No subjects passed the filtering criteria")
    print("  - Training phase had errors")
    print("  - Metrics computation had errors")
    print(f"\nDebug info:")
    print(f"  - Subjects trained: {len(results)}")
    print(f"  - Metrics computed: {len(advanced_metrics)}")
    print(f"  - Valid overlap: {len(set(results.keys()) & set(advanced_metrics.keys()))}")
else:
    print(f"\nTrain MSE:          {df_metrics['Train_MSE'].mean():.6f} ± {df_metrics['Train_MSE'].std():.6f}")
    print(f"Test MSE:           {df_metrics['Test_MSE'].mean():.6f} ± {df_metrics['Test_MSE'].std():.6f}")
    print(f"\nTrain TS Corr:      {df_metrics['Train_TS_Corr'].mean():.4f} ± {df_metrics['Train_TS_Corr'].std():.4f}")
    print(f"Test TS Corr:       {df_metrics['Test_TS_Corr'].mean():.4f} ± {df_metrics['Test_TS_Corr'].std():.4f}")
    print(f"\nTrain FC Corr:      {df_metrics['Train_FC_Corr'].mean():.4f} ± {df_metrics['Train_FC_Corr'].std():.4f}")
    print(f"Test FC Corr:       {df_metrics['Test_FC_Corr'].mean():.4f} ± {df_metrics['Test_FC_Corr'].std():.4f}")

## Visualizations

In [ ]:
# --- Compute FC metrics (empirical vs predicted) ---
print("\n" + "="*70)
print("COMPUTING FC METRICS")
print("="*70)

Corr_FC_emp_vs_sim = {}

torch.serialization.add_safe_globals([NPI.ANN_MLP, NPI.ANN_CNN, NPI.ANN_RNN, NPI.ANN_VAR])

for sub_id in sorted(per_subject_data.keys()):
    if sub_id not in results:
        continue
    
    print(f"\n{sub_id}:")
    
    # Load model
    model_path = os.path.join(results_dir, f"trained_models_{method}", f"{sub_id}_{method}.pt")
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    
    # Get test data
    X_test = per_subject_data[sub_id]['test_inputs']
    Y_test = per_subject_data[sub_id]['test_targets']
    
    # Predict
    with torch.no_grad():
        X_test_t = torch.tensor(X_test, dtype=torch.float32, device=device)
        Y_test_pred = model(X_test_t).cpu().numpy()
    
    # Compute FC matrices
    FC_emp = np.corrcoef(Y_test.T)
    FC_pred = np.corrcoef(Y_test_pred.T)
    
    # Extract upper triangle
    tri_idx = np.triu_indices(ROI_num, k=1)
    fc_emp_vec = FC_emp[tri_idx]
    fc_pred_vec = FC_pred[tri_idx]
    
    # Correlation
    r = np.corrcoef(fc_emp_vec, fc_pred_vec)[0, 1]
    Corr_FC_emp_vs_sim[sub_id] = r if not np.isnan(r) else 0.0
    
    print(f"  FC correlation: {r:.4f}")
    
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Computed FC metrics for {len(Corr_FC_emp_vs_sim)} subjects")


In [ ]:
# --- Compute dFC variance metrics (fluidity) ---
print("\n" + "="*70)
print("COMPUTING DFC VARIANCE METRICS")
print("="*70)

def go_edge(tseries):
    """Compute edge time series as product of z-scored signals."""
    nregions = tseries.shape[1]
    iTriup = np.triu_indices(nregions, k=1)
    gz = stats.zscore(tseries, axis=0)
    Eseries = gz[:, iTriup[0]] * gz[:, iTriup[1]]
    return Eseries

def compute_fluidity(tseries):
    """Compute dFC fluidity (variance of edge time series)."""
    try:
        Eseries = go_edge(tseries)
        fluidity = np.var(Eseries)
        return float(fluidity)
    except:
        return np.nan

Fluidity_emp = {}
Fluidity_sim = {}

for sub_id in sorted(per_subject_data.keys()):
    if sub_id not in results:
        continue
    
    print(f"\n{sub_id}:")
    
    # Load model
    model_path = os.path.join(results_dir, f"trained_models_{method}", f"{sub_id}_{method}.pt")
    model = torch.load(model_path, map_location=device, weights_only=False)
    model.eval()
    
    # Get test data
    Y_test = per_subject_data[sub_id]['test_targets']
    X_test = per_subject_data[sub_id]['test_inputs']
    
    # Empirical fluidity
    fluidity_emp = compute_fluidity(Y_test)
    Fluidity_emp[sub_id] = fluidity_emp
    
    # Simulated fluidity: generate autonomous time series
    S = using_steps
    N = ROI_num
    init_state = Y_test[:using_steps].copy()
    
    # Generate simulated time series using model
    Y_sim = []
    state = init_state.copy()
    
    with torch.no_grad():
        for t in range(Y_test.shape[0]):
            x = torch.tensor(state.reshape(-1).astype(np.float32), device=device).unsqueeze(0)
            y_pred = model(x).cpu().numpy().squeeze()
            Y_sim.append(y_pred)
            state = np.vstack([state[1:], y_pred])
    
    Y_sim = np.array(Y_sim)
    fluidity_sim = compute_fluidity(Y_sim)
    Fluidity_sim[sub_id] = fluidity_sim
    
    print(f"  Empirical fluidity: {fluidity_emp:.6f}")
    print(f"  Simulated fluidity: {fluidity_sim:.6f}")
    print(f"  Error: {abs(fluidity_sim - fluidity_emp) / (fluidity_emp + 1e-8) * 100:.2f}%")
    
    del model
    gc.collect()
    torch.cuda.empty_cache()

print(f"\n✅ Computed dFC fluidity for {len(Fluidity_emp)} subjects")


In [ ]:
# --- Plot FC Correlation Histogram ---
print("\n" + "="*70)
print("PLOTTING FC AND DFC METRICS")
print("="*70)

if len(Corr_FC_emp_vs_sim) > 1:
    data = [v for v in Corr_FC_emp_vs_sim.values() if not np.isnan(v)]
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Plot 1: FC correlation histogram
    ax = axes[0]
    ax.hist(data, bins=8, color='steelblue', edgecolor='black', alpha=0.7)
    ax.axvline(np.median(data), color='orange', linestyle='--', lw=2, label=f'Median: {np.median(data):.3f}')
    ax.axvline(np.mean(data), color='red', linestyle='--', lw=2, label=f'Mean: {np.mean(data):.3f}')
    ax.set_xlabel('FC Correlation (Empirical vs Predicted)', fontsize=11)
    ax.set_ylabel('Number of Subjects')
    ax.set_title('FC Correlation Distribution', fontsize=11, weight='bold')
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    
    # Plot 2: Ranked FC correlations
    ax = axes[1]
    subs_sorted = sorted(Corr_FC_emp_vs_sim.keys(), key=lambda x: Corr_FC_emp_vs_sim[x], reverse=True)
    fc_vals = [Corr_FC_emp_vs_sim[s] for s in subs_sorted]
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(subs_sorted)))
    ax.barh(range(len(subs_sorted)), fc_vals, color=colors, edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(subs_sorted)))
    ax.set_yticklabels(subs_sorted, fontsize=8)
    ax.set_xlabel('FC Correlation', fontsize=11)
    ax.set_title('Subjects Ranked by FC Correlation', fontsize=11, weight='bold')
    ax.invert_yaxis()
    ax.axvline(np.mean(data), color='red', linestyle='--', lw=2, alpha=0.5, label='Mean')
    ax.grid(axis='x', alpha=0.3)
    ax.legend(fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "FC_correlation_histogram.png"), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ FC correlation histogram saved")
    print(f"   Median FC correlation: {np.median(data):.4f}")
    print(f"   Mean FC correlation: {np.mean(data):.4f}")


In [ ]:
# --- Plot dFC Variance Scatter ---

if len(Fluidity_emp) > 1:
    subs = sorted(set(Fluidity_emp.keys()) & set(Fluidity_sim.keys()))
    
    x = np.array([Fluidity_emp[s] for s in subs], dtype=float)  # empirical
    y = np.array([Fluidity_sim[s] for s in subs], dtype=float)  # model
    
    # Regression
    from scipy.stats import linregress, spearmanr
    lr = linregress(x, y)
    r_spearman, p_spearman = spearmanr(x, y)
    
    fig, ax = plt.subplots(figsize=(5, 5))
    
    # Scatter
    ax.scatter(x, y, s=80, color='steelblue', edgecolor='white', 
               linewidth=1, alpha=0.8, zorder=3)
    
    # Regression line
    xx = np.linspace(x.min(), x.max(), 200)
    ax.plot(xx, lr.slope * xx + lr.intercept, color='steelblue',
            linewidth=2, linestyle="--", alpha=0.9, zorder=2)
    
    # Correlation annotation
    ax.text(0.05, 0.95, f"$r_S$ = {r_spearman:.2f}\n$p$ = {p_spearman:.2g}",
            transform=ax.transAxes, va='top', ha='left', fontsize=11,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="none", alpha=0.8))
    
    # Style
    ax.set_xlabel("Empirical dFC Variance (Fluidity)", fontsize=12, weight='bold')
    ax.set_ylabel("Model dFC Variance (Fluidity)", fontsize=12, weight='bold')
    ax.set_title("dFC Variance: Empirical vs Model", fontsize=12, weight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(alpha=0.3, zorder=0)
    
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "dFC_variance_scatter.png"), dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"✅ dFC variance scatter plot saved")
    print(f"   Spearman r: {r_spearman:.4f}, p-value: {p_spearman:.2g}")


In [ ]:
# --- Build enhanced metrics table with FC and dFC ---
print("\n" + "="*70)
print("BUILDING COMPREHENSIVE METRICS TABLE")
print("="*70)

enhanced_metrics = []

for sub_id in sorted(results.keys()):
    if sub_id not in per_subject_data:
        continue
    
    row = {
        'Subject': sub_id,
        'N_train': int(results[sub_id]['n_train']),
        'N_test': int(results[sub_id]['n_test']),
        'N_sessions': int(results[sub_id]['n_sessions']),
        'Train_MSE': float(results[sub_id]['final_train_loss']),
        'Test_MSE': float(results[sub_id]['final_test_loss']),
    }
    
    # Add FC correlation if available
    if sub_id in Corr_FC_emp_vs_sim:
        row['FC_Corr'] = float(Corr_FC_emp_vs_sim[sub_id])
    else:
        row['FC_Corr'] = np.nan
    
    # Add dFC fluidity metrics if available
    if sub_id in Fluidity_emp and sub_id in Fluidity_sim:
        row['Fluidity_Emp'] = float(Fluidity_emp[sub_id])
        row['Fluidity_Sim'] = float(Fluidity_sim[sub_id])
        row['Fluidity_Error_%'] = float(abs(Fluidity_sim[sub_id] - Fluidity_emp[sub_id]) / (Fluidity_emp[sub_id] + 1e-8) * 100)
    else:
        row['Fluidity_Emp'] = np.nan
        row['Fluidity_Sim'] = np.nan
        row['Fluidity_Error_%'] = np.nan
    
    enhanced_metrics.append(row)

df_enhanced = pd.DataFrame(enhanced_metrics)

# Display
print("\n" + df_enhanced.to_string(index=False))

# Save to CSV
csv_path = os.path.join(results_dir, "comprehensive_metrics_with_FC_dFC.csv")
df_enhanced.to_csv(csv_path, index=False)
print(f"\n✅ Saved comprehensive metrics to: {csv_path}")

# Summary statistics
print("\n" + "="*70)
print("SUMMARY STATISTICS")
print("="*70)
print(f"\nTraining Performance:")
print(f"  Train MSE:  mean={df_enhanced['Train_MSE'].mean():.6f} ± {df_enhanced['Train_MSE'].std():.6f}")
print(f"  Test MSE:   mean={df_enhanced['Test_MSE'].mean():.6f} ± {df_enhanced['Test_MSE'].std():.6f}")

if 'FC_Corr' in df_enhanced.columns and not df_enhanced['FC_Corr'].isna().all():
    print(f"\nFC Correlation (Empirical vs Predicted):")
    print(f"  Mean: {df_enhanced['FC_Corr'].mean():.4f} ± {df_enhanced['FC_Corr'].std():.4f}")
    print(f"  Range: [{df_enhanced['FC_Corr'].min():.4f}, {df_enhanced['FC_Corr'].max():.4f}]")

if 'Fluidity_Emp' in df_enhanced.columns and not df_enhanced['Fluidity_Emp'].isna().all():
    print(f"\ndFC Fluidity (Variance):")
    print(f"  Empirical: mean={df_enhanced['Fluidity_Emp'].mean():.6f} ± {df_enhanced['Fluidity_Emp'].std():.6f}")
    print(f"  Simulated: mean={df_enhanced['Fluidity_Sim'].mean():.6f} ± {df_enhanced['Fluidity_Sim'].std():.6f}")
    print(f"  Error: mean={df_enhanced['Fluidity_Error_%'].mean():.1f}% ± {df_enhanced['Fluidity_Error_%'].std():.1f}%")


In [ ]:
# --- Per-Subject Ranked Analysis Visualization ---
print("\n" + "="*70)
print("PER-SUBJECT RANKED ANALYSIS")
print("="*70)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Test MSE ranked (lower = better)
ax = axes[0, 0]
df_mse_sorted = df_enhanced.dropna(subset=['Test_MSE']).sort_values('Test_MSE', ascending=True).reset_index(drop=True)
subjects_mse = df_mse_sorted['Subject'].values
x_pos = np.arange(len(subjects_mse))
colors_mse = plt.cm.RdYlGn_r(np.linspace(0.2, 0.8, len(subjects_mse)))  # Reversed (green=low loss)
ax.barh(x_pos, df_mse_sorted['Test_MSE'].values, color=colors_mse, edgecolor='black', linewidth=0.5)
ax.set_yticks(x_pos)
ax.set_yticklabels(subjects_mse, fontsize=9)
ax.set_xlabel('Test MSE Loss')
ax.set_title('Subjects Ranked by Test Loss (Lower = Better)', weight='bold')
ax.invert_yaxis()
ax.axvline(df_enhanced['Test_MSE'].mean(), color='red', linestyle='--', lw=2, alpha=0.5, label='Mean')
ax.grid(axis='x', alpha=0.3)
ax.legend(fontsize=9)

# Plot 2: FC Correlation ranked (higher = better)
ax = axes[0, 1]
if not df_enhanced['FC_Corr'].isna().all():
    df_fc_sorted = df_enhanced.dropna(subset=['FC_Corr']).sort_values('FC_Corr', ascending=False).reset_index(drop=True)
    subjects_fc = df_fc_sorted['Subject'].values
    x_pos_fc = np.arange(len(subjects_fc))
    colors_fc = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(subjects_fc)))
    ax.barh(x_pos_fc, df_fc_sorted['FC_Corr'].values, color=colors_fc, edgecolor='black', linewidth=0.5)
    ax.set_yticks(x_pos_fc)
    ax.set_yticklabels(subjects_fc, fontsize=9)
    ax.set_xlabel('FC Correlation')
    ax.set_title('Subjects Ranked by FC Correlation (Higher = Better)', weight='bold')
    ax.invert_yaxis()
    ax.axvline(df_enhanced['FC_Corr'].mean(), color='red', linestyle='--', lw=2, alpha=0.5, label='Mean')
    ax.grid(axis='x', alpha=0.3)
    ax.legend(fontsize=9)
else:
    ax.text(0.5, 0.5, 'No FC data available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('FC Correlation Ranking')

# Plot 3: Train vs Test loss scatter
ax = axes[1, 0]
scatter = ax.scatter(df_enhanced['Train_MSE'], df_enhanced['Test_MSE'], 
                     s=100, c=range(len(df_enhanced)), cmap='viridis',
                     edgecolor='black', linewidth=0.8, alpha=0.7)
ax.set_xlabel('Train MSE Loss')
ax.set_ylabel('Test MSE Loss')
ax.set_title('Train vs Test Loss Comparison', weight='bold')
ax.grid(alpha=0.3)
# Add diagonal (perfect train=test)
lim = max(df_enhanced['Train_MSE'].max(), df_enhanced['Test_MSE'].max())
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=1)

# Plot 4: dFC Fluidity scatter (if available)
ax = axes[1, 1]
if not df_enhanced['Fluidity_Emp'].isna().all():
    scatter2 = ax.scatter(df_enhanced['Fluidity_Emp'], df_enhanced['Fluidity_Sim'],
                          s=100, c=range(len(df_enhanced)), cmap='plasma',
                          edgecolor='black', linewidth=0.8, alpha=0.7)
    ax.set_xlabel('Empirical dFC Fluidity')
    ax.set_ylabel('Simulated dFC Fluidity')
    ax.set_title('dFC Fluidity Comparison', weight='bold')
    ax.grid(alpha=0.3)
    # Add diagonal
    lim = max(df_enhanced['Fluidity_Emp'].max(), df_enhanced['Fluidity_Sim'].max())
    ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=1)
else:
    ax.text(0.5, 0.5, 'No dFC fluidity data available', ha='center', va='center', transform=ax.transAxes)
    ax.set_title('dFC Fluidity Comparison')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, "per_subject_ranked_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Per-subject ranked analysis saved")


In [ ]:
# --- Plot 1: Train vs Test Loss ---

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Learning curves (first 10 subjects)
ax = axes[0, 0]
for sub_id in sorted(results.keys())[:10]:
    train_loss = results[sub_id]["train_loss"]
    test_loss = results[sub_id]["test_loss"]
    ax.plot(train_loss, alpha=0.5, linewidth=1, label=f"{sub_id} (train)")
    ax.plot(test_loss, alpha=0.5, linewidth=1, linestyle="--", label=f"{sub_id} (test)")

ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.set_title("Learning Curves (First 10 Subjects)")
ax.grid(True, alpha=0.3)
ax.legend(fontsize=7, loc='upper right')

# 2. Final losses comparison
ax = axes[0, 1]
x = np.arange(len(df_metrics))
width = 0.35
ax.bar(x - width/2, df_metrics['Train_MSE'], width, label='Train', alpha=0.8, color='steelblue')
ax.bar(x + width/2, df_metrics['Test_MSE'], width, label='Test', alpha=0.8, color='coral')
ax.set_ylabel('Final MSE Loss')
ax.set_title('Train vs Test Loss (Final Epoch)')
ax.set_xticks(x)
ax.set_xticklabels(df_metrics['Subject'], rotation=45, ha='right', fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 3. Timeseries correlations (train vs test bars)
ax = axes[1, 0]
x = np.arange(len(df_metrics))
width = 0.35
ax.bar(x - width/2, df_metrics['Train_TS_Corr'], width, label='Train', alpha=0.8, color='steelblue')
ax.bar(x + width/2, df_metrics['Test_TS_Corr'], width, label='Test', alpha=0.8, color='coral')
ax.set_ylabel('Timeseries Correlation')
ax.set_title('Timeseries Correlation: Train vs Test')
ax.set_xticks(x)
ax.set_xticklabels(df_metrics['Subject'], rotation=45, ha='right', fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# 4. FC correlations (train vs test bars)
ax = axes[1, 1]
ax.bar(x - width/2, df_metrics['Train_FC_Corr'], width, label='Train', alpha=0.8, color='steelblue')
ax.bar(x + width/2, df_metrics['Test_FC_Corr'], width, label='Test', alpha=0.8, color='coral')
ax.set_ylabel('FC Correlation')
ax.set_title('FC Correlation: Train vs Test')
ax.set_xticks(x)
ax.set_xticklabels(df_metrics['Subject'], rotation=45, ha='right', fontsize=9)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(results_dir, "training_evaluation_rest_stim.png"), dpi=150, bbox_inches='tight')
plt.show()

print("✅ Saved training evaluation plot")

In [ ]:
# --- Plot 2: dFC Variance Correlation (DISABLED) ---
# dFC analysis has been commented out to save memory, so this plot is also disabled

print("⏭️  dFC variance plot skipped (dFC analysis disabled to save RAM)")

# Original code (commented out):
"""
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1. Empirical vs Simulated fluidity
ax = axes[0]
ax.scatter(df_metrics['Fluidity_Emp'], df_metrics['Fluidity_Sim'], s=100, alpha=0.7, edgecolor='black')
lim = max(df_metrics['Fluidity_Emp'].max(), df_metrics['Fluidity_Sim'].max())
ax.plot([0, lim], [0, lim], 'k--', alpha=0.3, linewidth=1)
ax.set_xlabel('Empirical Fluidity')
ax.set_ylabel('Simulated Fluidity')
ax.set_title('dFC Variance: Empirical vs Simulated')
ax.grid(alpha=0.3)
r_fluidity = np.corrcoef(df_metrics['Fluidity_Emp'], df_metrics['Fluidity_Sim'])[0, 1]
ax.text(0.05, 0.95, f'r = {r_fluidity:.3f}', transform=ax.transAxes, 
        fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# 2. Fluidity error distribution
ax = axes[1]
ax.hist(df_metrics['Fluidity_Error_%'], bins=10, color='steelblue', edgecolor='black', alpha=0.7)
ax.axvline(df_metrics['Fluidity_Error_%'].mean(), color='red', linestyle='--', lw=2,
          label=f"Mean: {df_metrics['Fluidity_Error_%'].mean():.1f}%")
ax.set_xlabel('Fluidity Error (%)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Fluidity Error')
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, "dfc_variance_correlation_rest_stim.png"), dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved dFC variance correlation plot")
"""

## Summary

**Key Results:**

1. **Training Data:** Combined rest + stim sessions with per-session 50/50 split
   - Cleaner train/test separation while allowing diverse training data
   - One unique model per subject (same format as before)
   - Excluded subjects with < 1000 total timepoints

2. **Evaluation Metrics:**
   - **Train/Test MSE Loss**: Measures prediction accuracy
   - **Timeseries Correlation**: Per-region prediction accuracy averaged across ROIs
   - **FC Correlation**: How well model predicts functional connectivity structure
   - **dFC Variance (Fluidity)**: Checks if simulated data has similar dynamic FC variability as empirical

3. **Models:**
   - Saved to: `trained_models_MLP/sub-XXX_MLP.pt` (same naming as before, no duplicates)
   - Each subject has one MLP trained on combined rest+stim data
   - Ready for downstream simulation and analysis

4. **Outputs:**
   - `metrics_rest_stim_combined.csv` — Full metrics table
   - `training_evaluation_rest_stim.png` — Learning curves and correlations
   - `dfc_variance_correlation_rest_stim.png` — dFC variance analysis